# Notebook 4 – Duplicate Data Handling
## Why Blindly Deleting Duplicates Can Be Dangerous

Duplicates *look* like an easy cleanup step — one line of code and they're gone.
But not every duplicate is "junk data." Sometimes a repeated row is real, valid,
and meaningful. Deleting it without checking *why* it repeated can quietly
delete real people, real events, or real signal from your dataset.

Let's see this using our own HR dataset.

In [1]:
import pandas as pd

df = pd.read_csv("hr_employee_attrition_raw.csv")
print("Total rows:", len(df))
print("Exact duplicate rows:", df.duplicated().sum())

Total rows: 1230
Exact duplicate rows: 20


### Quick Check: Who Are These Duplicates?

We have **20 exact duplicate rows**. Feels safe to drop them, right?
Let's actually look at one before deleting anything.

In [2]:
# Look at duplicates based on EmployeeID (a supposedly unique key)
dupes = df[df.duplicated(subset=["EmployeeID"], keep=False)].sort_values("EmployeeID")
dupes.head(6)

,EmployeeID,Age,Gender,Department,JobRole,Education,MonthlyIncome,YearsAtCompany,JobSatisfaction,PerformanceRating,DistanceFromHomeKM,OverTime,JoinDate,Attrition,EmployeeCountFlag,RandomSurveyCode
15,EMP10135,26,Female,Sales,Sales Manager,Bachelor's,4021.79,2,1.0,1,1.5,No,2018-12-28,No,1,564179
844,EMP10135,26,Female,Sales,Sales Manager,Bachelor's,4021.79,2,1.0,1,1.5,No,2018-12-28,No,1,564179
111,EMP10189,32,Female,HR,HR Manager,Bachelor's,4210.69 USD,7,3.0,2,1.0,No,2020-08-22,Yes,1,357934
1191,EMP10189,32,Female,HR,HR Manager,Bachelor's,4210.69 USD,7,3.0,2,1.0,No,2020-08-22,Yes,1,357934
1033,EMP10190,31,Female,HR,HR Executive,Bachelor's,4981.16,0,2.0,2,9.6,No,2015-05-02,Yes,1,471986
1129,EMP10190,31,Female,HR,HR Executive,Bachelor's,4981.16,0,2.0,2,9.6,No,2015-05-02,Yes,1,471986


### 1. A Duplicate Might Be a Real Repeated Event, Not an Error

If `EmployeeID` were something like `TransactionID` in a payroll or attendance
log, seeing the same ID twice could mean the employee was **paid twice, logged
in twice, or flagged twice for a genuine reason** - not that the row was
copy-pasted by mistake.

Blindly running `drop_duplicates()` here could silently erase a real event
(e.g., a bonus payment, a second training session, a second incident report)
just because it looked identical to another row.

### 2. "Duplicate" Depends on Which Columns You Check

Look at `EMP10189` below  - same `EmployeeID`, same `Attrition`, same income,
same everything **except** the income is written as `"4210.69 USD"` instead
of `4210.69`. A basic `df.duplicated()` on the whole row might **not** catch
this as a duplicate because one column doesn't match exactly.

So relying only on full-row duplicates can *miss* duplicates, while relying
only on `EmployeeID` can flag rows as duplicates that actually have different
values elsewhere. Neither method alone tells the full story - you have to
decide which columns actually define "this is the same record."

In [3]:
# Same person, but does everything actually match?
df[df["EmployeeID"] == "EMP10189"]

,EmployeeID,Age,Gender,Department,JobRole,Education,MonthlyIncome,YearsAtCompany,JobSatisfaction,PerformanceRating,DistanceFromHomeKM,OverTime,JoinDate,Attrition,EmployeeCountFlag,RandomSurveyCode
111,EMP10189,32,Female,HR,HR Manager,Bachelor's,4210.69 USD,7,3.0,2,1.0,No,2020-08-22,Yes,1,357934
1191,EMP10189,32,Female,HR,HR Manager,Bachelor's,4210.69 USD,7,3.0,2,1.0,No,2020-08-22,Yes,1,357934


### 3. You Could Lose the "Most Correct" Version of a Record

When two rows share an ID but one has a missing `Education` value and the
other doesn't, `drop_duplicates()` by default keeps the **first occurrence** - even if it's the row with less information. You might keep the worse
version of the record and throw away the better one, purely by chance of
row order.

In [4]:
# drop_duplicates() keeps the FIRST match by default — check what you're keeping
df.drop_duplicates(subset=["EmployeeID"], keep="first").loc[
    df["EmployeeID"].isin(["EMP10189"])
]

,EmployeeID,Age,Gender,Department,JobRole,Education,MonthlyIncome,YearsAtCompany,JobSatisfaction,PerformanceRating,DistanceFromHomeKM,OverTime,JoinDate,Attrition,EmployeeCountFlag,RandomSurveyCode
111,EMP10189,32,Female,HR,HR Manager,Bachelor's,4210.69 USD,7,3.0,2,1.0,No,2020-08-22,Yes,1,357934


### 4. Aggregate Numbers Get Skewed

If you're calculating average `MonthlyIncome` or attrition rate, dropping
duplicates the wrong way changes the sample size for the wrong reason:

- Drop too little - real duplicates inflate averages/counts.
- Drop too much - you lose real employees, and your % Attrition or average
  salary shifts based on data loss, not reality.

Either mistake changes conclusions someone might act on - like a manager
deciding to give raises or headcount based on a skewed number.

## The Simple Takeaway

> A duplicate is not automatically an "error." It's a **signal** that
> something is worth investigating - a data entry mistake, a system glitch,
> or a genuinely repeated event. Always ask **why** it duplicated before
> deciding **whether** to delete it.

**Business rule to apply here:** Two rows count as duplicates only if they
match on `EmployeeID` **and** every other field is identical after cleaning
(e.g., stripping "USD" from income). When keeping one row, keep the one with
the fewest missing values, not just the first one seen.

## A Safer Cleanup Approach

In [5]:
# A safer approach: clean first, then decide, then drop deliberately
df["MonthlyIncome"] = (
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False).astype(float)
)

# Now duplicates are real duplicates, not formatting mismatches
true_dupes = df.duplicated(subset=["EmployeeID"], keep=False)
print("Real duplicate rows after cleaning:", true_dupes.sum())

df_clean = df.drop_duplicates(subset=["EmployeeID"], keep="first")
print("Rows before:", len(df), "| Rows after:", len(df_clean))

Real duplicate rows after cleaning: 40
Rows before: 1230 | Rows after: 1210
